### Make dict with item url being the (normalized) URL, and the value being the bibtex citekey

Used for matching URL cites in perplexity dialogs to the bibext citekey filenames of obsidian notes.

In [1]:
# TODO: 
# - SPACE BEFORE LINKS IN MD DOC, 
# - SOME UNCLOSED '']'' NEAR eol
# - ORIG FOOTNOTES ALSO MISSING
# - RETAIN ORIGINAL CONTENTS, SO CAN CONSIDER ADDING NEW LINKS FROM IT LATER

In [2]:
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import sys
from urllib.parse import urlparse, urlunparse
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import re

%load_ext autoreload
%autoreload 2

In [3]:
tmp_dir = rfw.refwrangle_test_dir / 'tmp'

perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
obsidian_citekeys_file = rfw.refwrangle_test_dir / "dat" / 'obsnotecitekeys.csv'

output_file = tmp_dir / "tmp_new_cites_perplexity_example.md"

In [4]:
zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)
parentItems = zot.everything(zot.top())

In [10]:
# get the URLs of all parent items in the zotero db, and find out which have obsidian literature notes
citekeys = {}
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    citekeyThis = rfw.get_citation_key(pdat)

    if 'url' in pdat:
        purl = rfw.normalize_url(pdat['url'])
        if len(purl)>0:
            citekeysForURL[purl].append(citekeyThis)

repeatedURLs = {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}

if (nURLrepeats := len(repeatedURLs)) > 0:
    print(f"There were {nURLrepeats} URLs with > 1 parent (citekey)")
    for url in repeatedURLs.keys():
        print(f"{repeatedURLs[url]}\n\t{url}")
    raise Exception(f'Not built for repeated URLS: {nURLrepeats=}.')

url_to_citekey={}
for (key, value_list) in citekeysForURL.items():
    url_to_citekey[key] = value_list[0]

In [ ]:
lit_note_file_stems = {fNm.stem for fNm in rfw.lit_notes_obsidian_dir.glob('*.md')}
citekeys = {}
zot_db_items = []
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    citekeyThis = rfw.get_citation_key(pdat)
    zot_db_items.append(dict(citekey=citekeyThis, zotkey=parent['key'], hasLitNote=citekeyThis in lit_note_file_stems))

    if 'url' in pdat:
        purl = rfw.normalize_url(pdat['url'])
        if len(purl)>0:
            citekeysForURL[purl].append(citekeyThis)

repeatedURLs = {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}

if (nURLrepeats := len(repeatedURLs)) > 0:
    print(f"There were {nURLrepeats} URLs with > 1 parent (citekey)")
    for url in repeatedURLs.keys():
        print(f"{repeatedURLs[url]}\n\t{url}")
    raise Exception(f'Not built for repeated URLS: {nURLrepeats=}.')

url_to_citekey={}
citekey_to_url = {}
for (url, citekey_list) in citekeysForURL.items():
    url_to_citekey[url] = citekey_list[0]
    citekey_to_url[citekey_list[0]] = url


#pd.Series(citekey_to_url)
zot_db_items = pd.DataFrame(zot_db_items).set_index('citekey')
zot_db_items['url'] = pd.Series(citekey_to_url)
zot_db_items = zot_db_items.reset_index()

hasNoURL = zot_db_items.url.isna()
if (nURLmiss := sum(hasNoURL)) > 0:
    print(f"Dropping {nURLmiss=} of {len(zot_db_items)} zotero entries which have no URL")
    zot_db_items_no_url = zot_db_items[hasNoURL]
    zot_db_items = zot_db_items[~hasNoURL]
    display(zot_db_items_no_url)


In [ ]:
zot_db_items

In [128]:
def zotero_item_link(zotero_item_key, link_text):
    return f'[{link_text}](zotero://select/library/items/{zotero_item_key})'

def normalize_url(url):
    """Convert a URL to a standard form, so the it can be string-compared to the same URL
    written by a different program, but which is also normalized by this function."""
    parsed = urlparse(url.lower())
    return urlunparse(parsed._replace(path=parsed.path.rstrip('/')))

def replace_perplexity_citations_from_perplexity(markdown_file, zot_db_items, output_file):
    """Replace numeric citations in a perplexity dialog with any obsidian literature note links that are
    given either as a dict or a file."""

    # Read the CSV file and create a dictionary of normalized URL to citekey mappings
    if not isinstance(zot_db_items, pd.DataFrame):
        raise Exception('Expected a dataframe.  Reading url_to_citekey from file does not yet handle new dataframe column')
        df = pd.read_csv(zot_db_items) # assume it has url and citekey columns
        zot_db_items = {normalize_url(url): citekey for url, citekey in zip(df.url, df.citekey)}

    zot_db_items['url'] = zot_db_items['url'].apply(normalize_url)
    zot_url_to_item_info = defaultdict(lambda: None, {url:info.iloc[0] for url, info in zot_db_items.groupby('url')})

    with open(markdown_file, 'r') as mdfile:
        content = mdfile.read()

    # Split the content into body and citations
    parts = content.split("\nCitations:\n")
    if len(parts) != 2:
        raise Exception("Couldn't find Citations section")
    
    body, citations = parts

    # Extract citations and their corresponding normalized URLs
    citation_urls = re.findall(r'\[(\d+)\]\s+(https?://\S+)', citations)
    doc_number_to_url = defaultdict(lambda: None, {num:normalize_url(url) for num, url in citation_urls})

    # Replace citations in the body text with a wikilinks to an obsidian note, or to or zotero item
    def replace_citation(match):
        doc_cite_num = match.group(1)
        doc_url = doc_number_to_url[doc_cite_num]

        if doc_url:
            if (itemInfo := zot_url_to_item_info[doc_url]) is not None:
            #if itemInfo is not None:
                if itemInfo.hasLitNote:
                    return f' [[{itemInfo.citekey}]]'
                link_text = f'{itemInfo.zotkey}==>{itemInfo.citekey}'
                return f' {zotero_item_link(itemInfo.zotkey, link_text)}'
        return f' [{doc_cite_num}]'

    body = re.sub(r'\[(\d+)\]', replace_citation, body)

    # Replace citations in the Citations section
    def replace_citation_in_references(match):
        doc_cite_num = match.group(1)
        url = match.group(2)
        doc_url = normalize_url(url)

        matching_row = zot_db_items[zot_db_items.url == doc_url]
        if not matching_row.empty:
            return f'[[{matching_row.iloc[0].citekey}]] {doc_url}'
        return f'[{doc_cite_num}] {doc_url}'

    citations = re.sub(r'\[(\d+)\]\s+(https?://\S+)', replace_citation_in_references, citations)

    # Combine modified body and citations
    modified_content = body + "\nCitations:\n" + citations

    # Write the modified content to the output file
    with open(output_file, 'w') as outfile:
        outfile.write(modified_content)


In [129]:
replace_perplexity_citations_from_perplexity(perplexity_dialog_file, zot_db_items, output_file)
#rfw.ORIG_replace_perplexity_citations_from_perplexity(perplexity_dialog_file, url_to_citekey, output_file)
print('Done.')

Done.


In [130]:
#zot_db_items

In [131]:
#{url:info for url, info in zot_db_items.groupby('url')}['/paper/gaussian-process-regression-discontinuity-ornstein-duck-mayr/ff90d4584bd9e84ca918635c2c4cda654e6409a4']

doc_url_to_info = defaultdict(lambda: None, {url:info.iloc[0] for url, info in zot_db_items.groupby('url')})
#doc_url_to_info = defaultdict(lambda: None, {url:info.to_dict() for url, info in zot_db_items.groupby('url')})
#doc_url_to_info = defaultdict(lambda: None, {url:pd.Series(info) for url, info in zot_db_items.groupby('url')})

z = doc_url_to_info['/paper/gaussian-process-regression-discontinuity-ornstein-duck-mayr/ff90d4584bd9e84ca918635c2c4cda654e6409a4']
type(pd.Series(z))


pandas.core.series.Series

In [132]:
if z.hasLitNote:
    print('true')

In [133]:
row_dicts = [row.to_dict() for _, row in zot_db_items.iterrows()]
row_dicts

#row_dicts = {url:info.to_dict() for url, info in zot_db_items.iterrows()}
row_dicts = {url:info.to_dict() for url, info in zot_db_items.iterrows()}
row_dicts

{0: {'citekey': 'Krysiak-Adamczyk25brandSentimAnlyss',
  'zotkey': '839Z6XEL',
  'hasLitNote': False,
  'url': 'https://survicate.com/blog/brand-sentiment-analysis'},
 1: {'citekey': 'Wolanin24brandSentimentCare',
  'zotkey': '7A9XQEFG',
  'hasLitNote': False,
  'url': 'https://brand24.com/blog/brand-sentiment'},
 2: {'citekey': 'Mohamed2monitorToolsPR',
  'zotkey': 'MW8W5LIA',
  'hasLitNote': False,
  'url': 'https://www.aimtechnologies.co/pr-monitoring-tools-enhancing-your-public-relations-strategy'},
 3: {'citekey': 'Mohamed23sentimAnlyssTools',
  'zotkey': 'XZ927L88',
  'hasLitNote': False,
  'url': 'https://www.aimtechnologies.co/sentiment-analysis-tools-unveiling-insights-from-textual-data'},
 4: {'citekey': 'Mathew21socialMediaToolsTop23',
  'zotkey': 'K3EGPXPI',
  'hasLitNote': False,
  'url': 'https://www.meltwater.com/en/blog/top-social-media-monitoring-tools'},
 5: {'citekey': 'Newton23importanceBrandPercept',
  'zotkey': 'WPXSY5NK',
  'hasLitNote': False,
  'url': 'https://